Read input data

In [1]:
from pandas_plink import read_plink
import pandas as pd

## Plink files
(bim, fam, G) = read_plink("../tests/data/EUR.QC", verbose=False)

## Read base data
base_data = pd.read_csv("../tests/data/Height.QC.gz", sep="\t", compression="gzip")

## Read covariate data
covariate_data = pd.read_csv("../tests/data/EUR.covariate", sep=" ")

/tmp/ipykernel_1049/1365574468.py:5: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  (bim, fam, G) = read_plink("../tests/data/EUR.QC", verbose=False)
/tmp/ipykernel_1049/1365574468.py:5: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  (bim, fam, G) = read_plink("../tests/data/EUR.QC", verbose=False)


In [2]:
command = {
    "binary-target": "F",
    "base-maf": "MAF:0.01",
    "base-info": "INFO:0.8",
    "stat": "OR",
    "or": True,
    "out": "EUR"
}

In [3]:
bim

,chrom,snp,cm,pos,a0,a1,i
0,1,rs3131962,0.490722,756604,A,G,0
1,1,rs4040617,0.500708,779322,G,A,1
2,1,rs79373928,0.587220,801536,G,T,2
3,1,rs11240779,0.620827,808631,G,A,3
4,1,rs57181708,0.620827,809876,G,A,4
...,...,...,...,...,...,...,...
489800,22,rs73174435,75.082500,51174939,T,C,489800
489801,22,rs3810648,75.083200,51175626,G,A,489801
489802,22,rs5771002,75.089100,51183255,A,G,489802
489803,22,rs3865764,75.091100,51185848,G,A,489803


In [ ]:
BITCT = 64
BITCT2 = BITCT / 2
VEC_BYTES = 16
VEC_BITS = VEC_BYTES * 8
VEC_WORDS = VEC_BITS / BITCT

BITCT_TO_VECCT = lambda val: (((val) + (VEC_BITS - 1)) / VEC_BITS)
BITCT_TO_ALIGNED_WORDCT = lambda val: VEC_WORDS * BITCT_TO_VECCT(val)
BITCT_TO_WORDCT = lambda val: (((val) + (BITCT - 1)) / BITCT)
QUATERCT_TO_VECCT = lambda val: (((val) + ((VEC_BITS / 2) - 1)) / (VEC_BITS / 2))

QUATERCT_TO_WORDCT = lambda val: (((val) + (BITCT2 - 1)) / BITCT2)
QUATERCT_TO_ALIGNED_WORDCT = lambda val: (VEC_WORDS * QUATERCT_TO_VECCT(val))


class Genotype:

    def __init__(self, plink_files, base_file, cov_file):
        self.bim, self.fam, self.G = read_plink(plink_files, verbose=False)
        self.base_data = pd.read_csv(base_file, sep="\t", compression="gzip")
        self.covariate_data = pd.read_csv(cov_file, sep=" ")

        # inc/storage.hpp:193
        self.clumping_struct = {
            "distance": 250000
        }

    @property
    def m_sort_by_p_index(self):
        """
        Sort by: chr ↑ → p_value ↑ → loc ↑ → rs ↑
        """
        # inc/genotype.hpp:139
        idx = self.bim.merge(
                    self.base_data[["SNP", "P", "CHR", "BP"]], 
                    left_on="snp", 
                    right_on="SNP", 
                    how="left")\
                .sort_values(by=["CHR", "P", "BP", "SNP"])\
                .index.to_list()                # src/snp.cpp:27
        
        return idx

    def get_chrom_boundary(self):
        # src/genotype.cpp:1171
        dta = self.bim.loc[:,"chrom"].value_counts()\
                        .reset_index()\
                        .astype({"chrom": int})\
                        .sort_values(by="chrom")
        dta["second"] = dta["count"].cumsum()
        dta["first"] = dta["second"].shift(1, fill_value=0)
        dta_list = dta.loc[:,["first", "second"]].values.tolist()

        return dta_list

    @property
    def m_max_window_size(self):
        # src/genotype.cpp:99 & src/genotype.cpp:111
        # src/genotype.cpp:83
        m_max_window_size = 0
        prev_chr = 0
        low_bound = 0
        prev_loc = 0
        cur_dist = 0

        for i_dx, (chr, pos) in enumerate(bim[["chrom", "pos"]].values):
            ## Get max gap between chromosomes (number of snp)
            if (prev_chr != chr):
                prev_chr = chr
                prev_loc = pos
                low_bound = i_dx
            snp_distance = i_dx - low_bound
            if (m_max_window_size < snp_distance): 
                m_max_window_size = snp_distance
            
            ## Lift up low_bound by limiting clumping distance (position distance)
            cur_dist = pos - prev_loc
            while(cur_dist > self.clumping_struct["distance"] and low_bound < i_dx):
                snp_distance = i_dx - low_bound
                low_bound+=1
                prev_loc = bim["pos"][low_bound]
                cur_dist = pos - prev_loc
            
            if (m_max_window_size < snp_distance): 
                m_max_window_size = snp_distance
        
        return m_max_window_size

    def clumping(self):
        # src/genotype.cpp:1194
        # src/genotype.cpp:1274
        min_r2 = 0.1
        m_founder_ct = len(self.fam)
        m_unfiltered_sample_ct = len(self.fam)
        founder_ctv3 = BITCT_TO_ALIGNED_WORDCT(m_founder_ct)
        founder_ctl2 = QUATERCT_TO_WORDCT(m_founder_ct)

        founder_ctsplit = 3 * founder_ctv3
        founder_ctv2 = QUATERCT_TO_ALIGNED_WORDCT(m_founder_ct)
        unfiltered_sample_ctl = BITCT_TO_WORDCT(m_unfiltered_sample_ct)

        unfiltered_sample_ctv2 = 2 * unfiltered_sample_ctl
        index_data = [0] * 3 * founder_ctsplit + founder_ctv3
        index_tots = [0] * 6

        founder_include2 = [0] * founder_ctv2

        snp_range = self.get_chrom_boundary()            # src/genotype.cpp:1200

        num_snp_in_chr = sum([i[1] - i[0] for i in snp_range])
        max_snp_in_chr = 0

        # src/genotype.cpp:1300
        max_size = max(self.m_max_window_size, num_snp_in_chr * 0.01)

        pass

gt = Genotype(
        "../tests/data/EUR.QC",
        "../tests/data/Height.QC.gz",
        "../tests/data/EUR.covariate"
    )

gt.m_max_window_size

/tmp/ipykernel_1049/235326157.py:19: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  self.bim, self.fam, self.G = read_plink(plink_files, verbose=False)
/tmp/ipykernel_1049/235326157.py:19: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  self.bim, self.fam, self.G = read_plink(plink_files, verbose=False)


547